# Step 5 — 移動回測與自動評估
模擬真實預測情境：逐日移動訓練窗口，計算出手率與準確率

Big Data & Business Analytics, National Taiwan University

In [1]:
# ── 可調整參數區 ──────────────────────────────────────
WINDOW_DAYS  = 30    # 訓練窗口天數（使用前 N 個日曆天的文章訓練）
MIN_ARTICLES = 5     # 當日有效文章篇數門檻，低於此值不出手
SIGNAL_RATIO = 0.2   # 出手信號門檻：|看漲數-看跌數| / 總數 < 此值則不出手
# ──────────────────────────────────────────────────────

In [2]:
import csv
import pickle
from datetime import datetime, timedelta
from collections import Counter, defaultdict
import numpy as np
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, accuracy_score

# [METHOD] 目前方法：30天滾動窗口 | 可替換為：60天、週為單位

In [3]:
# 載入 feature_list（來自 step3）
f = open('vectors.pkl', 'rb')
data = pickle.load(f)
f.close()
feature_list = data['feature_list']
print(f'特徵詞數量：{len(feature_list)}')

特徵詞數量：200


In [4]:
# 讀入 articles_labeled.csv
articles = []
f = open('articles_labeled.csv', newline='', encoding='utf-8')
reader = csv.DictReader(f)
for row in reader:
    row['label'] = int(row['label'])
    try:
        row['post_dt']   = datetime.strptime(row['post_time'], '%Y-%m-%d %H:%M:%S')
        row['post_date'] = row['post_dt'].date()
        articles.append(row)
    except Exception:
        pass
f.close()

print(f'讀入 {len(articles)} 篇文章')

讀入 4350 篇文章


In [5]:
# N-gram 切割與向量化函式（與 step3 相同）
def get_ngrams(text, min_n=2, max_n=4):
    ngrams = []
    for n in range(min_n, max_n + 1):
        for i in range(len(text) - n + 1):
            gram = text[i:i+n]
            if gram.strip():
                ngrams.append(gram)
    return ngrams

def vectorize(text, feature_list):
    gram_count = Counter(get_ngrams(text))
    return [gram_count.get(f, 0) for f in feature_list]

In [6]:
# 依日期分組
by_date = defaultdict(list)
for art in articles:
    by_date[art['post_date']].append(art)

all_dates = sorted(by_date.keys())
print(f'共 {len(all_dates)} 個有文章的日期')

共 603 個有文章的日期


In [7]:
# 移動回測主迴圈
results = []   # 每次出手的結果：(日期, 預測方向, 實際方向)

for test_date in all_dates:

    # ── 收集訓練資料（前 WINDOW_DAYS 個日曆天，排除中性文章）──
    window_start = test_date - timedelta(days=WINDOW_DAYS)
    train_arts = [
        a
        for d, arts in by_date.items()
        if window_start <= d < test_date
        for a in arts
        if a['label'] != 0
    ]

    # ── 收集當日有效文章（排除中性）──
    test_arts = [a for a in by_date[test_date] if a['label'] != 0]

    # 門檻 1：當日有效文章不足
    if len(test_arts) < MIN_ARTICLES:
        continue

    # 門檻 2：訓練集需同時包含看漲與看跌文章
    train_labels = [a['label'] for a in train_arts]
    if len(train_arts) < MIN_ARTICLES or 1 not in train_labels or -1 not in train_labels:
        continue

    # ── 向量化訓練集 ──
    X_train = []
    y_train = []
    for art in train_arts:
        text = (art.get('title','') or '') + ' ' + (art.get('content','') or '')
        X_train.append(vectorize(text, feature_list))
        y_train.append(art['label'])

    # ── 向量化測試集 ──
    X_test = []
    for art in test_arts:
        text = (art.get('title','') or '') + ' ' + (art.get('content','') or '')
        X_test.append(vectorize(text, feature_list))

    # ── 訓練並預測 ──
    model = MultinomialNB()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    n_up   = sum(1 for p in y_pred if p ==  1)
    n_down = sum(1 for p in y_pred if p == -1)
    total  = n_up + n_down

    if total == 0:
        continue

    # 門檻 3：信號不夠強（看漲/看跌比例差距 < SIGNAL_RATIO）
    if abs(n_up - n_down) / total < SIGNAL_RATIO:
        continue

    # ── 決定預測方向 ──
    pred_dir = 1 if n_up > n_down else -1

    # ── 決定實際方向（當日文章的多數標籤）──
    actual_up   = sum(1 for a in test_arts if a['label'] ==  1)
    actual_down = sum(1 for a in test_arts if a['label'] == -1)
    actual_dir  = 1 if actual_up >= actual_down else -1

    results.append((test_date, pred_dir, actual_dir))

print(f'回測完成，出手 {len(results)} 次')

回測完成，出手 173 次


In [9]:
# 輸出回測結果
n_total = len(all_dates)
n_trade = len(results)
trade_rate = n_trade / n_total * 100 if n_total > 0 else 0

print('===== 移動回測結果 =====')
print(f'總文章天數：{n_total} 天')
print(f'出手天數：{n_trade} 天（出手率：{trade_rate:.1f}%）')

if n_trade > 0:
    y_actual    = [r[2] for r in results]
    y_predicted = [r[1] for r in results]

    cm  = confusion_matrix(y_actual, y_predicted, labels=[1, -1])
    acc = accuracy_score(y_actual, y_predicted)

    print()
    print('Confusion Matrix：')
    print(f'{"":12s} 預測漲    預測跌')
    print(f'真實漲    |  {cm[0][0]:4d}  |  {cm[0][1]:4d}  |')
    print(f'真實跌    |  {cm[1][0]:4d}  |  {cm[1][1]:4d}  |')
    print()
    print(f'總準確率：{acc*100:.1f}%')
else:
    print('（沒有符合出手條件的日期）')

===== 移動回測結果 =====
總文章天數：603 天
出手天數：173 天（出手率：28.7%）

Confusion Matrix：
             預測漲    預測跌
真實漲    |    50  |    50  |
真實跌    |    31  |    42  |

總準確率：53.2%


In [10]:
# 逐日預測明細（前 20 筆）
print('\n逐日預測明細（前20筆）：')
print(f'{"日期":12s}  {"預測":4s}  {"實際":4s}  {"結果":4s}')
print('-' * 35)
for date, pred, actual in results[:20]:
    pred_str   = '漲' if pred   ==  1 else '跌'
    actual_str = '漲' if actual ==  1 else '跌'
    ok         = '✓' if pred == actual else '✗'
    print(f'{str(date):12s}  {pred_str:4s}  {actual_str:4s}  {ok:4s}')


逐日預測明細（前20筆）：
日期            預測    實際    結果  
-----------------------------------
2023-03-09    跌     跌     ✓   
2023-03-10    跌     跌     ✓   
2023-03-30    跌     漲     ✗   
2023-04-06    跌     漲     ✗   
2023-04-07    跌     漲     ✗   
2023-04-24    漲     跌     ✗   
2023-04-28    漲     漲     ✓   
2023-05-02    跌     跌     ✓   
2023-05-04    漲     漲     ✓   
2023-05-10    漲     漲     ✓   
2023-05-15    漲     漲     ✓   
2023-05-16    漲     漲     ✓   
2023-05-17    漲     漲     ✓   
2023-05-18    漲     漲     ✓   
2023-05-19    漲     漲     ✓   
2023-05-26    漲     漲     ✓   
2023-05-31    漲     漲     ✓   
2023-06-08    漲     跌     ✗   
2023-06-15    漲     漲     ✓   
2023-06-20    漲     跌     ✗   
